# Introduzione alla Modellazione Semantica con dbt

Questo notebook accompagna la lezione sulla **modellazione semantica** usando dbt e il database AdventureWorks.

## Cos'è la modellazione semantica?

In un contesto analytics, la **modellazione semantica** include due livelli che si completano:

- **Modelli dbt (mart)** — Tabelle o viste con grain e metriche già calcolate in modo corretto (es. `fct_orders`, `fct_customers`), così query semplici restano affidabili.
- **Semantic layer + MetricFlow** — Definizioni dichiarative in YAML (semantic models, misure, metriche) che descrivono entità, dimensioni e aggregazioni riusabili; MetricFlow (`mf`) valida e compila le query sulle metriche.

**Il problema**: senza una buona modellazione, ogni analista rischia gli stessi errori (fanout, campi sbagliati, filtri dimenticati). Centralizzare la logica nei modelli e nel layer semantico la rende **corretta by design** e **riusabile** da BI e altri client.

## Obiettivi della lezione

1. **Esplorare i dati grezzi** — Capire la struttura dei dati sorgente
2. **Comprendere il problema del fanout** — Vedere come un JOIN può "gonfiare" i conteggi
3. **Vedere come dbt risolve questi problemi** — Modelli pre-calcolati e testabili
4. **Confrontare query sbagliate vs corrette** — Imparare dai propri errori
5. **Layer semantico e MetricFlow** — Grain, entità, dimensioni, misure, metriche; metriche su `fct_orders` e `mf query`

## Setup iniziale

Prima di iniziare, assicurati di aver eseguito `uv sync` nella root del progetto per installare le dipendenze (DuckDB, dbt-duckdb, Jupyter). Se hai già fatto `dbt seed` e `dbt run` nella cartella `adventureworks/`, il database sarà pronto.

In [ ]:
# Installazione dipendenze (se necessario)
# !uv sync

## 1. Esplorazione dei Dati Grezzi

Iniziamo esplorando i dati grezzi. I dati sono memorizzati in file CSV nella cartella `adventureworks/seeds/` e vengono caricati nel database DuckDB tramite `dbt seed`.

**Perché partire dai dati grezzi?** Perché la modellazione semantica parte sempre dalla comprensione della sorgente. Se non conosci la struttura delle tabelle, non puoi progettare modelli corretti.

Ci connettiamo al database DuckDB e elenchiamo le tabelle disponibili. Il database viene creato da dbt nella cartella `adventureworks/data/`.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

# Path DB: cwd tipico = root repo oppure notebooks/
_cwd = Path.cwd()
_db = _cwd / "adventureworks" / "data" / "adventureworks.duckdb"
if not _db.is_file():
    _db = _cwd.parent / "adventureworks" / "data" / "adventureworks.duckdb"
if not _db.is_file():
    raise FileNotFoundError(
        f"Non trovo adventureworks.duckdb (provato sotto {_cwd} e {_cwd.parent}). "
        "Esegui dbt da adventureworks/ e avvia Jupyter dalla root del repo o da notebooks/."
    )

# Path assoluto: serve alla cella MetricFlow per riconnettersi dopo mf query.
DUCKDB_PATH = str(_db.resolve())

# Sola lettura per le SELECT del notebook; la cella mf chiude comunque `con` prima di lanciare mf.
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# Lista delle tabelle
print("Tabelle disponibili:")
tables = con.execute("SHOW TABLES").fetchall()
for t in tables:
    print(f"  - {t[0]}")

Tabelle disponibili:
  - categories
  - customers
  - fct_customers
  - fct_orders
  - fct_products
  - order_lines
  - orders
  - products
  - stg_categories
  - stg_customers
  - stg_order_lines
  - stg_orders
  - stg_products
  - time_spine_daily


### 1.1 Clienti

La tabella **customers** contiene i clienti con i loro dati anagrafici. Nota il campo `city`: lo useremo per segmentazioni geografiche. Ogni cliente ha un `customer_id` univoco che collega agli ordini.

In [2]:
# Visualizza clienti
customers = con.execute("SELECT * FROM customers").df()
print("CLIENTI:")
print(customers.to_string(index=False))

CLIENTI:
 customer_id first_name last_name               email   city
         101       John       Doe    john@example.com  Milan
         102       Jane     Smith    jane@example.com   Rome
         103        Bob   Johnson     bob@example.com Naples
         104      Alice  Williams   alice@example.com  Milan
         105    Charlie     Brown charlie@example.com   Rome


### 1.2 Ordini

La tabella **orders** contiene gli ordini. Il campo `status` è numerico: **1** = pending, **5** = shipped, **6** = cancelled. Per le analisi di revenue useremo solo gli ordini shipped (status = 5). Il campo `total` è il totale a livello ordine — ma attenzione: potrebbe non riflettere gli sconti applicati alle singole righe!

In [3]:
# Visualizza ordini
orders = con.execute("SELECT * FROM orders").df()
print("ORDINI (status: 1=pending, 5=shipped, 6=cancelled):")
print(orders.to_string(index=False))

ORDINI (status: 1=pending, 5=shipped, 6=cancelled):
 order_id  customer_id order_date  total  status
        1          101 2024-01-15   1500       5
        2          102 2024-01-16   2200       5
        3          101 2024-01-17    800       5
        4          103 2024-01-18   3100       5
        5          101 2024-01-19    450       5
        6          104 2024-01-20   1800       1
        7          105 2024-01-21    950       2
        8          102 2024-02-01   1650       5
        9          104 2024-02-05   4200       5
       10          103 2024-02-10    750       6


### 1.3 Righe Ordine (con sconti!)

La tabella **order_lines** è la più importante per le metriche. Ogni riga rappresenta un prodotto in un ordine. Il campo `line_total` è il totale lordo; `discount_pct` è la percentuale di sconto (es. 0.10 = 10%). Il revenue netto si calcola come `line_total * (1 - discount_pct)`.

**Regola d'oro**: per metriche accurate, calcola sempre dalle righe di dettaglio, non dai totali dell'header.

In [4]:
# Visualizza righe ordine
order_lines = con.execute("SELECT * FROM order_lines").df()
print("RIGHE ORDINE (nota: discount_pct = percentuale sconto, es. 0.10 = 10%):")
print(order_lines.to_string(index=False))

RIGHE ORDINE (nota: discount_pct = percentuale sconto, es. 0.10 = 10%):
 order_line_id  order_id  product_id  quantity  unit_price  line_total  discount_pct
             1         1           1         2         500        1000          0.00
             2         1           2         1         600         600          0.10
             3         2           3        10          50         500          0.00
             4         2           1         2         500        1000          0.15
             5         3           2         1         600         600          0.00
             6         3           3         4          50         200          0.00
             7         4           4         1        2500        2500          0.20
             8         4           2         1         600         600          0.00
             9         5           1         1         500         500          0.00
            10         6           1         2         500        1000        

## 2. Il Problema del Fanout

Il **fanout** è uno dei problemi più comuni nelle query SQL su dati transazionali. Succede quando fai un JOIN tra tabelle e poi un'aggregazione senza cure adeguate.

**Come funziona il fanout?** Quando un ordine ha più righe (es. 3 prodotti), il JOIN con `order_lines` "espande" ogni ordine in 3 righe. Se poi fai `COUNT(order_id)` senza DISTINCT, conti 3 invece di 1. Il risultato è **gonfiato**.

**Soluzione**: usare `COUNT(DISTINCT order_id)` per contare gli ordini unici. Vediamo la differenza nelle prossime celle.

**Query sbagliata**: qui contiamo `ol.order_id` — ma ogni ordine appare tante volte quante sono le righe in `order_lines`. Un ordine con 3 prodotti viene contato 3 volte. Osserva i numeri: sono troppo alti!

In [5]:
# Esempio: Query sbagliata - conta le righe, non gli ordini
query_sbagliata = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(ol.order_id) AS order_count  -- ERRORE! Conta righe
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_sbagliato = con.execute(query_sbagliata).df()
print("❌ QUERY SBAGLIATA (fanout):")
print(result_sbagliato.to_string(index=False))

❌ QUERY SBAGLIATA (fanout):
 customer_id  customer_name  order_count
         101       John Doe            5
         104 Alice Williams            5
         102     Jane Smith            4
         103    Bob Johnson            3
         105  Charlie Brown            2


**Query corretta**: aggiungiamo `DISTINCT` a `COUNT(DISTINCT o.order_id)`. Ora ogni ordine viene contato una sola volta, indipendentemente da quante righe ha. I numeri sono realistici.

In [6]:
# Query corretta - usa DISTINCT
query_corretta = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(DISTINCT o.order_id) AS order_count  -- CORRETTO!
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_corretto = con.execute(query_corretta).df()
print("\n⚠️ QUERY CORRETTA (con DISTINCT):")
print(result_corretto.to_string(index=False))


⚠️ QUERY CORRETTA (con DISTINCT):
 customer_id  customer_name  order_count
         101       John Doe            3
         104 Alice Williams            2
         102     Jane Smith            2
         103    Bob Johnson            2
         105  Charlie Brown            1


**Confronto side-by-side**: la colonna `differenza` mostra quanto abbiamo "gonfiato" i conteggi con la query sbagliata. Più righe per ordine = più errore.

In [14]:
# Confronto: togliamo customer_name dal df di destra, altrimenti pandas crea customer_name_sbagliato / _corretto
print("\n📊 DIFFERENZA:")
comparison = result_sbagliato.merge(
    result_corretto.drop(columns=["customer_name"]),
    on="customer_id",
    suffixes=("_sbagliato", "_corretto"),
)
comparison["differenza"] = comparison["order_count_sbagliato"] - comparison["order_count_corretto"]
print(comparison[["customer_name", "order_count_sbagliato", "order_count_corretto", "differenza"]].to_string(index=False))


📊 DIFFERENZA:
 customer_name  order_count_sbagliato  order_count_corretto  differenza
      John Doe                      5                     3           2
    Jane Smith                      4                     2           2
   Bob Johnson                      3                     2           1
 Charlie Brown                      2                     1           1
Alice Williams                      5                     2           3


## 3. Problema 2: Sconti e Revenue

Un altro problema comune: usare il campo `total` dell'ordine invece di calcolare il revenue dalle righe (dove ci sono gli sconti).

**Perché è un problema?** Il campo `orders.total` è spesso un valore denormalizzato che potrebbe non includere gli sconti applicati alle singole righe. La **fonte di verità** per il revenue è sempre `order_lines`.

**Query sbagliata** (cella successiva): usiamo `SUM(o.total)` dall'header. Questo valore potrebbe non riflettere gli sconti applicati alle singole righe.

Eseguite la cella sottostante per vedere la query sbagliata. Userà `SUM(o.total)` dall'header — notate che i valori potrebbero non riflettere gli sconti.

Ecco il codice della query sbagliata. Notate `SUM(o.total)` — non considera gli sconti delle singole righe.

In [15]:
# Query sbagliata: usa il campo total dell'header
query_header = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    SUM(o.total) AS lifetime_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 5  -- solo shipped
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_header = con.execute(query_header).df()
print("❌ QUERY SBAGLIATA (usa orders.total):")
print(result_header.to_string(index=False))

❌ QUERY SBAGLIATA (usa orders.total):
 customer_id  customer_name  lifetime_value
         101       John Doe          2750.0
         103    Bob Johnson          3100.0
         102     Jane Smith          3850.0
         104 Alice Williams          4200.0


**Query corretta**: calcoliamo il revenue da `order_lines` con `line_total * (1 - discount_pct)`. Ogni riga ha il proprio sconto. Eseguite la cella sottostante.

In [16]:
# Query corretta: calcola da righe con sconti
query_lines = """
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    SUM(ol.line_total * (1 - ol.discount_pct)) AS lifetime_value  -- considera sconti!
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
WHERE o.status = 5
GROUP BY c.customer_id, c.first_name, c.last_name
"""

result_lines = con.execute(query_lines).df()
print("\n⚠️ QUERY CORRETTA (calcola da order_lines con sconti):")
print(result_lines.to_string(index=False))


⚠️ QUERY CORRETTA (calcola da order_lines con sconti):
 customer_id  customer_name  lifetime_value
         101       John Doe          2840.0
         104 Alice Williams          4200.0
         103    Bob Johnson          2600.0
         102     Jane Smith          4300.0


**Confronto**: la colonna `differenza` mostra quanto "perdiamo" in revenue quando usiamo il totale dell'header invece di calcolare dalle righe. Gli sconti fanno la differenza!

In [17]:
# Confronto: stesso trucco — evitare due colonne customer_name dopo il merge
print("\n📊 DIFFERENZA (gli sconti fanno la differenza!):")
comp = result_header.merge(
    result_lines.drop(columns=["customer_name"]),
    on="customer_id",
    suffixes=("_header", "_lines"),
)
comp["differenza"] = comp["lifetime_value_header"] - comp["lifetime_value_lines"]
print(comp[["customer_name", "lifetime_value_header", "lifetime_value_lines", "differenza"]].to_string(index=False))


📊 DIFFERENZA (gli sconti fanno la differenza!):
 customer_name  lifetime_value_header  lifetime_value_lines  differenza
      John Doe                 2750.0                2840.0       -90.0
   Bob Johnson                 3100.0                2600.0       500.0
    Jane Smith                 3850.0                4300.0      -450.0
Alice Williams                 4200.0                4200.0         0.0


## 4. I Modelli dbt in Azione

Ora vediamo come i modelli dbt risolvono questi problemi **automaticamente**. I modelli mart (`fct_*`) sono tabelle pre-calcolate che applicano già:

- **Filtro** `status = 5` (solo ordini shipped)
- **COUNT(DISTINCT ...)** dove serve per evitare il fanout
- **Calcolo del revenue** dalle righe con sconti (`discount_pct`)

Non devi più ricordare queste regole: sono **incapsulate** nel modello. Dopo `dbt run`, le fact table sono pronte e affidabili.

### fct_customers

La fact table dei clienti contiene già `order_count` (con DISTINCT), `lifetime_value` (da header, per confronto) e `lifetime_net_revenue` (calcolato dalle righe con sconti). Tutto filtrato per ordini shipped.

In [7]:
# Visualizza i modelli marts
print("📦 FACT TABLES CREATE DA DBT:\n")

# fct_customers
fct_customers = con.execute("SELECT * FROM fct_customers").df()
print("fct_customers:")
print(fct_customers.to_string(index=False))

📦 FACT TABLES CREATE DA DBT:

fct_customers:
 customer_id first_name last_name               email   city  order_count  lifetime_value  lifetime_net_revenue
         104      Alice  Williams   alice@example.com  Milan            1         12600.0                4200.0
         105    Charlie     Brown charlie@example.com   Rome            0             0.0                   0.0
         103        Bob   Johnson     bob@example.com Naples            1          6200.0                2600.0
         102       Jane     Smith    jane@example.com   Rome            2          7700.0                4300.0
         101       John       Doe    john@example.com  Milan            3          5050.0                2840.0


### fct_products

Analisi a livello prodotto: quanti ordini contengono ogni prodotto (`orders_with_product`), unità vendute, revenue lorda e netta. Anche qui `COUNT(DISTINCT order_id)` evita il fanout.

In [8]:
# fct_products
fct_products = con.execute("SELECT * FROM fct_products").df()
print("\nfct_products:")
print(fct_products.to_string(index=False))


fct_products:
 product_id    product_name  category_id  subcategory_id category_name subcategory_name  orders_with_product  units_sold  gross_revenue  net_revenue
          4 Time Trial Bike            1               3         Bikes Time Trial Bikes                    3         3.0         7500.0       7000.0
          2   Mountain Bike            1               2         Bikes   Mountain Bikes                    5         6.0         3600.0       3540.0
          3          Helmet            2               4   Accessories          Helmets                    5        41.0         2050.0       2050.0
          1       Road Bike            1               1         Bikes       Road Bikes                    6         9.0         4500.0       4300.0
          5       Bike Lock            2               5   Accessories            Locks                    0         0.0            0.0          0.0


### fct_orders

Analisi a livello ordine: totale header vs somma delle righe (gross e net). Utile per reconciliazione e per capire se i dati sono allineati.

In [9]:
# fct_orders
fct_orders = con.execute("SELECT * FROM fct_orders").df()
print("\nfct_orders:")
print(fct_orders.to_string(index=False))


fct_orders:
 order_id order_date  customer_id  customer_name   city  status  order_total  line_count  total_items  gross_revenue  net_revenue
        2 2024-01-16          102     Jane Smith   Rome       5         2200           2         12.0         1500.0       1350.0
        5 2024-01-19          101       John Doe  Milan       5          450           1          1.0          500.0        500.0
        9 2024-02-05          104 Alice Williams  Milan       5         4200           3          4.0         4200.0       4200.0
        8 2024-02-01          102     Jane Smith   Rome       5         1650           2          2.0         3000.0       2950.0
        4 2024-01-18          103    Bob Johnson Naples       5         3100           2          2.0         3100.0       2600.0
        3 2024-01-17          101       John Doe  Milan       5          800           2          5.0          800.0        800.0
        1 2024-01-15          101       John Doe  Milan       5         1500 

## 5. Il Vantaggio della Modellazione Semantica (mart dbt)

I mart dbt garantiscono che:

1. ✅ **Filtri corretti** — Solo ordini shipped (status = 5), escludendo pending e cancelled
2. ✅ **Aggregazioni corrette** — COUNT(DISTINCT ...) dove serve, niente fanout
3. ✅ **Metriche accurate** — Calcolo dai dettagli (order_lines), non dai campi aggregati
4. ✅ **Logica centralizzata** — Un solo punto di modifica nei modelli SQL

**Risultato**: gli analisti possono scrivere query semplici su `fct_*` senza rischiare errori.

Il passo successivo è il **semantic layer** (file YAML nel progetto dbt): stesse idee, ma con **nome metrica**, **dimensioni** e **tempo** espliciti, consumabili con **MetricFlow** senza riscrivere l’aggregazione a mano ogni volta.

In [21]:
# Esempio: query semplice come dovrebbe essere
print("✅ Con dbt, per ottenere i clienti per valore:")
print("\nSELECT first_name, last_name, lifetime_net_revenue")
print("FROM fct_customers")
print("ORDER BY lifetime_net_revenue DESC;\n")

simple = con.execute("SELECT first_name, last_name, lifetime_net_revenue FROM fct_customers ORDER BY lifetime_net_revenue DESC").df()
print("RISULTATO:")
print(simple.to_string(index=False))

✅ Con dbt, per ottenere i clienti per valore:

SELECT first_name, last_name, lifetime_net_revenue
FROM fct_customers
ORDER BY lifetime_net_revenue DESC;

RISULTATO:
first_name last_name  lifetime_net_revenue
      Jane     Smith                4300.0
     Alice  Williams                4200.0
      John       Doe                2840.0
       Bob   Johnson                2600.0
   Charlie     Brown                   0.0


**Esempio pratico**: per ottenere i clienti ordinati per valore, basta una query di 3 righe. Nessun JOIN, nessun GROUP BY, nessun rischio di fanout. La complessità è nascosta nel modello. Eseguite la cella sopra per vedere il risultato!

## 6. Semantic layer e MetricFlow

Il file `adventureworks/models/marts/_semantic_layer.yml` definisce un **semantic model** `orders_semantic` sulla fact `fct_orders`. Nel README (sezione *Livello 4*, glossario) trovi le definizioni estese; qui il riassunto legato a questo file:

- **Grain** — Cosa rappresenta una riga: in `fct_orders` è **un ordine** (shipped). Il layer non cambia il grain: lo eredita dalla tabella.
- **Semantic model** — Il blocco YAML che lega `ref('fct_orders')` a entità, dimensioni e misure (`orders_semantic`).
- **Entity** — Chiavi nel grafo semantico: `order` (primary, `order_id`) e `customer` (foreign, `customer_id`) per relazioni future tra modelli.
- **Dimension** — Attributi per tagliare i numeri: `order_date` (tempo, granularità giorno), `city`, `customer_name`.
- **Measure** — Aggregazione sul modello, es. `sum` di `net_revenue`; per i conteggi qui usiamo `count_distinct` su `order_id` (`orders_count`), più esplicito di `sum(1)` (vedi README, nota conteggi).
- **Metric** — Nome “pubblico” usato in query. Oltre alle `simple` (es. `total_net_revenue` → measure `net_revenue`), in `_semantic_layer.yml` c’è **`avg_net_revenue_per_order`** con `type: ratio`: rapporto tra le metriche `total_net_revenue` e `order_count` (AOV netto).
- **Time spine** — Tabella calendario giornaliera (`time_spine_daily`) richiesta dal semantic layer per il tempo.

### Perché MetricFlow se abbiamo già le fact?

Le fact (`fct_orders`, …) sono il posto giusto dove **materializzare** grain, filtri e colonne corrette: restano la base per SQL libero e per i test dbt.

**MetricFlow** non sostituisce quelle tabelle: aggiunge un **contratto** sulle **metriche** (nomi, misure, `ratio`, tempo) così non ogni report riscrive `SUM(net_revenue)` e `GROUP BY` a mano. Valida le definizioni (`mf validate-configs`) e genera SQL coerente con il grafo semantico. Su un dataset piccolo come questo, `SELECT SUM(net_revenue) FROM fct_orders` basta spesso; in azienda, con molti consumatori e tool diversi, avere **una sola definizione** di `total_net_revenue` ripaga. Dettaglio in README (*Livello 4*, tabella “MetricFlow vs query dirette sulle fact”).

In **`mf query --group-by`** spesso servono nomi **qualificati** `entità__dimensione` (es. `order__city`): se usi solo `city`, la CLI può suggerire la forma corretta nell’errore.

Dalla **root del repository** (dove c’è `pyproject.toml`), la cella sotto esegue `mf query` con `DBT_PROFILES_DIR` sulla root. In alternativa: terminale in `adventureworks/` con `DBT_PROFILES_DIR=..` come nel README.

**Lock DuckDB**: MetricFlow/dbt aprono il file in scrittura. La cella sotto **chiude temporaneamente** `con`, lancia `mf`, poi **riconnette** con `DUCKDB_PATH` (definito nella cella DuckDB). La connessione resta `read_only=True` per le sole SELECT; senza chiusura, DuckDB segnala *Conflicting lock* tra kernel Jupyter e il processo `mf`.

In [10]:
import os
import subprocess
import sys
from pathlib import Path

import duckdb


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / "profiles.yml").is_file() and (d / "adventureworks" / "dbt_project.yml").is_file():
            return d
    base = cwd.name
    if base in ("notebooks", "adventureworks"):
        return cwd.parent
    return cwd


def _mf_cmd_prefix() -> list[str]:
    """Usa `mf` dello stesso venv del kernel se c'è; altrimenti `uv run mf`."""
    bindir = Path(sys.executable).resolve().parent
    mf = bindir / ("mf.exe" if sys.platform == "win32" else "mf")
    if mf.is_file():
        return [str(mf)]
    return ["uv", "run", "mf"]


def mf_query(extra_args: list[str]) -> None:
    """MetricFlow: cwd=adventureworks, DBT_PROFILES_DIR=root. Chiude `con` per evitare lock DuckDB col processo mf."""
    global con
    root = str(_repo_root())
    aw = os.path.join(root, "adventureworks")
    env = os.environ.copy()
    env["DBT_PROFILES_DIR"] = root
    cmd = [*_mf_cmd_prefix(), "query", *extra_args]

    db_path = globals().get("DUCKDB_PATH")
    reopen = db_path and globals().get("con") is not None
    if reopen:
        try:
            con.close()
        except Exception:
            pass
        con = None

    try:
        r = subprocess.run(cmd, cwd=aw, capture_output=True, text=True, env=env)
        if r.stdout:
            print(r.stdout, end="")
        if r.stderr:
            print(r.stderr, end="")
        if r.returncode != 0:
            raise RuntimeError(
                f"mf query fallito (exit {r.returncode}). Comando: {' '.join(cmd)}\n"
                "Suggerimenti: kernel del progetto (.venv), DBT_PROFILES_DIR, e aver eseguito le celle fino a DUCKDB_PATH."
            )
    finally:
        if reopen and db_path:
            con = duckdb.connect(db_path, read_only=True)


print("--- total_net_revenue ---")
mf_query(["--metrics", "total_net_revenue", "--quiet"])

print("--- per order__city ---")
mf_query(["--metrics", "total_net_revenue", "--group-by", "order__city"])

print("--- avg_net_revenue_per_order (metrica ratio, non simple) ---")
mf_query(["--metrics", "avg_net_revenue_per_order", "--quiet"])

--- total_net_revenue ---
  total_net_revenue
-------------------
              13940
--- per order__city ---


⠋ Initiating query…

⠙ Initiating query…
✔ Success 🦄 - query completed after 0.15 seconds
order__city      total_net_revenue
-------------  -------------------
Rome                          4300
Naples                        2600
Milan                         7040

--- avg_net_revenue_per_order (metrica ratio, non simple) ---
  avg_net_revenue_per_order
---------------------------
                    1991.43


## Esercizi Proposti

Per consolidare i concetti, provate questi esercizi. **Suggerimento**: scrivete prima la query in SQL diretto (sulle tabelle raw), poi rifatela usando i modelli dbt. Confrontate la complessità!

### Base (con i modelli)

1. **Prodotti per categoria**: Quanti prodotti unici sono stati venduti per categoria? Usa `fct_products` e raggruppa per `category_name`.
2. **Revenue per città**: Quale città ha generato più revenue? Usa `fct_customers` e raggruppa per `city`.
3. **Top clienti**: Chi sono i 3 clienti con il maggior `lifetime_net_revenue`?
4. **Prodotti più venduti**: Quali sono i 3 prodotti con più `units_sold`? Usa `fct_products`.

### Intermedi (query su tabelle raw)

5. **Sconto medio per prodotto**: Qual è il prodotto con lo sconto medio più alto? Usa `order_lines` e calcola `AVG(discount_pct)` per `product_id`.
6. **Reconciliazione**: Confronta `order_total` e `gross_revenue` in `fct_orders`. Ci sono ordini dove differiscono?
7. **Ordini per stato**: Quanti ordini ci sono per ogni `status`? Usa la tabella `orders`.

### Avanzati (modifiche al progetto dbt)

8. **Nuovo mart giornaliero**: Crea `fct_daily_sales` con revenue aggregata per giorno.
9. **Test di integrità**: Aggiungi un test dbt che verifica che `gross_revenue` ≈ `order_total` in `fct_orders`.

### Semantic layer / MetricFlow

10. **`mf query --explain`**: Dalla cartella `adventureworks/`, confronta la SQL generata per `total_net_revenue` con `SELECT SUM(net_revenue) FROM fct_orders`.
11. **Nuova metrica**: In `_semantic_layer.yml` aggiungi una metrica `simple` su `gross_revenue`, poi `dbt parse` e `mf validate-configs`.

Buon lavoro!

---

**Fine della lezione.** Ricordate di chiudere la connessione al database quando avete finito.

In [ ]:
# Chiudi connessione
con.close()
print("✅ Notebook completato!")